# 01 — Exploratory Data Analysis: MVTec AD (6-Category Subset)

This notebook explores the 6-category subset uploaded to Kaggle:
**bottle, capsule, carpet, hazelnut, leather, pill**

In [ ]:
import sys, pathlib
# Ensure the project root is on the path
PROJECT_ROOT = pathlib.Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import csv
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

In [ ]:
# ── Configuration ──────────────────────────────────────────
MVTEC_ROOT = Path(r'd:/A_Online_courses/ML DEPI/Project/mvtec-ad')
CATEGORIES = ['bottle', 'capsule', 'carpet', 'hazelnut', 'leather', 'pill']
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'

## 1. Per-Category Image Counts

In [ ]:
summary = {}
for cat in CATEGORIES:
    cat_dir = MVTEC_ROOT / cat
    train_good = len(list((cat_dir / 'train' / 'good').glob('*.png')))
    test_good = 0
    test_defect = 0
    defect_types = {}
    for d in sorted((cat_dir / 'test').iterdir()):
        if not d.is_dir():
            continue
        n = len(list(d.glob('*.png')))
        if d.name == 'good':
            test_good = n
        else:
            test_defect += n
            defect_types[d.name] = n
    summary[cat] = {
        'train_normal': train_good,
        'test_normal': test_good,
        'test_defect': test_defect,
        'defect_types': defect_types,
        'total': train_good + test_good + test_defect,
    }

# Display as table
import pandas as pd
df_summary = pd.DataFrame([
    {'Category': k, 'Train (normal)': v['train_normal'],
     'Test (normal)': v['test_normal'], 'Test (defect)': v['test_defect'],
     'Total': v['total']}
    for k, v in summary.items()
])
df_summary

## 2. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: total images per category
cats = list(summary.keys())
train_n = [summary[c]['train_normal'] for c in cats]
test_n = [summary[c]['test_normal'] for c in cats]
test_d = [summary[c]['test_defect'] for c in cats]

x = np.arange(len(cats))
w = 0.25
axes[0].bar(x - w, train_n, w, label='Train (normal)', color='#2ecc71')
axes[0].bar(x,     test_n,  w, label='Test (normal)',  color='#3498db')
axes[0].bar(x + w, test_d,  w, label='Test (defect)',  color='#e74c3c')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cats, rotation=30)
axes[0].set_ylabel('Image Count')
axes[0].set_title('Image Count by Category & Split')
axes[0].legend()

# Pie chart: overall label ratio
total_normal = sum(v['train_normal'] + v['test_normal'] for v in summary.values())
total_defect = sum(v['test_defect'] for v in summary.values())
axes[1].pie([total_normal, total_defect],
            labels=['Normal', 'Anomalous'],
            autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'],
            startangle=90)
axes[1].set_title('Overall Label Distribution')

plt.tight_layout()
plt.savefig('docs/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Defect Type Breakdown

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, cat in zip(axes.flat, CATEGORIES):
    dt = summary[cat]['defect_types']
    ax.barh(list(dt.keys()), list(dt.values()), color='#e74c3c', alpha=0.8)
    ax.set_title(cat, fontweight='bold')
    ax.set_xlabel('Count')
plt.suptitle('Defect Types per Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/defect_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sample Image Grid

In [ ]:
fig, axes = plt.subplots(len(CATEGORIES), 5, figsize=(15, 3 * len(CATEGORIES)))

for row, cat in enumerate(CATEGORIES):
    # Show 3 normal + 2 defect samples
    normal_dir = MVTEC_ROOT / cat / 'train' / 'good'
    normal_imgs = sorted(normal_dir.glob('*.png'))[:3]
    
    test_dir = MVTEC_ROOT / cat / 'test'
    defect_imgs = []
    for d in sorted(test_dir.iterdir()):
        if d.is_dir() and d.name != 'good':
            imgs = sorted(d.glob('*.png'))
            if imgs:
                defect_imgs.append(imgs[0])
        if len(defect_imgs) >= 2:
            break
    
    all_imgs = normal_imgs + defect_imgs
    labels = ['Normal'] * len(normal_imgs) + ['Defect'] * len(defect_imgs)
    
    for col in range(5):
        ax = axes[row, col]
        if col < len(all_imgs):
            img = Image.open(all_imgs[col]).convert('RGB')
            ax.imshow(img)
            color = 'green' if labels[col] == 'Normal' else 'red'
            ax.set_title(f'{labels[col]}', color=color, fontsize=10)
        ax.axis('off')
    axes[row, 0].set_ylabel(cat, fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Sample Images per Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Image Resolution & Channel Statistics

In [ ]:
stats = []
for cat in CATEGORIES:
    normal_dir = MVTEC_ROOT / cat / 'train' / 'good'
    imgs = sorted(normal_dir.glob('*.png'))[:20]  # sample 20 per category
    
    widths, heights, means, stds = [], [], [], []
    for p in imgs:
        img = np.array(Image.open(p).convert('RGB'))
        h, w, c = img.shape
        widths.append(w)
        heights.append(h)
        means.append(img.mean(axis=(0, 1)) / 255.0)
        stds.append(img.std(axis=(0, 1)) / 255.0)
    
    stats.append({
        'Category': cat,
        'Width': f"{np.mean(widths):.0f}",
        'Height': f"{np.mean(heights):.0f}",
        'Mean (R,G,B)': f"({np.mean([m[0] for m in means]):.3f}, {np.mean([m[1] for m in means]):.3f}, {np.mean([m[2] for m in means]):.3f})",
        'Std (R,G,B)': f"({np.mean([s[0] for s in stds]):.3f}, {np.mean([s[1] for s in stds]):.3f}, {np.mean([s[2] for s in stds]):.3f})",
    })

df_stats = pd.DataFrame(stats)
df_stats

## 6. Mean Pixel Intensity per Category

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cat_means = {}
for cat in CATEGORIES:
    normal_dir = MVTEC_ROOT / cat / 'train' / 'good'
    imgs = sorted(normal_dir.glob('*.png'))[:30]
    pixel_means = [np.array(Image.open(p).convert('RGB')).mean() / 255.0 for p in imgs]
    cat_means[cat] = pixel_means

ax.boxplot(cat_means.values(), labels=cat_means.keys())
ax.set_ylabel('Mean Pixel Intensity (0-1)')
ax.set_title('Pixel Intensity Distribution by Category')
plt.tight_layout()
plt.savefig('docs/pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Split CSV Verification

In [ ]:
# Verify split CSVs exist and show row counts
split_info = []
for csv_path in sorted(SPLITS_DIR.glob('*.csv')):
    if csv_path.name.startswith('.'):
        continue
    counts = Counter()
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            counts[row['split']] += 1
    split_info.append({
        'File': csv_path.name,
        'Train': counts.get('train', 0),
        'Val': counts.get('val', 0),
        'Test': counts.get('test', 0),
        'Total': sum(counts.values()),
    })

df_splits = pd.DataFrame(split_info)
df_splits

## 8. Dataset & DataLoader Smoke Test

In [ ]:
from src.data.mvtec_dataset import MVTecDataset
from src.data.augmentations import get_eval_transform

ds = MVTecDataset(
    root=str(MVTEC_ROOT),
    category='bottle',
    split_csv=str(SPLITS_DIR / 'bottle_ratio70_15_15_seed42.csv'),
    transform=get_eval_transform(),
    split_filter='train',
)

sample = ds[0]
print(f'Dataset length: {len(ds)}')
print(f'Sample type:    {type(sample).__name__}')
print(f'Image shape:    {sample.image.shape}')
print(f'Image dtype:    {sample.image.dtype}')
print(f'Label:          {sample.label}')
print(f'Category:       {sample.category}')
print(f'Path:           {sample.path}')